In [7]:
import random
import math
import numpy as np
from copy import deepcopy

def team_average(team):
    return sum(team) / len(team)

def compute_energy(teams):
    avgs = [team_average(team) for team in teams]
    return max(avgs) - min(avgs)

def get_team_sizes(N, K):
    base = N // K
    rem = N % K
    return [base + 1 if i < rem else base for i in range(K)]

def build_initial_teams(captains, goalies, others):
    K = len(captains)
    sizes = get_team_sizes(len(captains) + len(goalies) + len(others), K)
    teams = [[captains[i], goalies[i]] for i in range(K)]
    random.shuffle(others)

    # Fill remaining players using round-robin into teams with space
    fill_slots = [s - 2 for s in sizes]  # already have captain + goalie
    idx = 0
    for i in range(K):
        teams[i].extend(others[idx:idx + fill_slots[i]])
        idx += fill_slots[i]

    return teams

def propose_swap_variable(teams):
    i, j = random.sample(range(len(teams)), 2)
    if len(teams[i]) < 3 or len(teams[j]) < 3:
        return None  # can't swap if either team has only captain/goalie
    pi = random.randint(2, len(teams[i]) - 1)
    pj = random.randint(2, len(teams[j]) - 1)
    
    new_teams = deepcopy(teams)
    new_teams[i][pi], new_teams[j][pj] = new_teams[j][pj], new_teams[i][pi]
    return new_teams

def simulated_annealing_variable(captains, goalies, others, T_start=1.0, T_end=1e-3, alpha=0.995, max_iters=100000, c = 0):
    current = build_initial_teams(captains, goalies, others)
    current_energy = compute_energy(current)
    best = deepcopy(current)
    best_energy = current_energy
    T = T_start

    for step in range(max_iters):
        if T < T_end:
            break

        candidate = propose_swap_variable(current)
        if candidate is None:
            continue

        candidate_energy = compute_energy(candidate)
        delta = candidate_energy - current_energy

        if delta < 0 or random.random() < math.exp(-delta / T):
            current = candidate
            current_energy = candidate_energy
            if current_energy < best_energy:
                best = deepcopy(current)
                best_energy = current_energy

        T *= alpha

    return best, best_energy

In [14]:
def sort_teams(teams):
    """Sort the 'others' in each team and return the full list of sorted teams."""
    sorted_teams = []

    for team in teams:
        captain = team[0]
        goalie = team[1]
        others = team[2:]

        # Sort the others explicitly
        sorted_others = []
        for val in sorted(others):
            sorted_others.append(val)

        # Rebuild team
        new_team = [captain, goalie]
        for val in sorted_others:
            new_team.append(val)

        sorted_teams.append(new_team)

    return sorted_teams

In [15]:
from itertools import permutations

def best_goalie_permutation_sa(captains, goalies, others, **sa_kwargs):
    best_teams = None
    best_energy = float('inf')
    best_perm = None
    c = 0
    for perm in permutations(goalies):
        teams, energy = simulated_annealing_variable(
            captains=captains,
            goalies=list(perm),
            others=others,
            c = c,
            **sa_kwargs
        )
        #Print results
        if(c%100 == 0):
            teams = sort_teams(teams)
            print("Energy:", round(compute_energy(teams), 7))
            for i, team in enumerate(teams):
                print(f"Team {i+1} avg: {team_average(team):.3f}  -->  {team}")
        c+=1
        if energy < best_energy:
            best_energy = energy
            best_teams = teams
            best_perm = perm

    return best_teams, best_energy, best_perm

In [16]:
K = 6
captains = [5, 4, 5, 4, 3, 3]
goalies = [4, 3, 6, 4, 2, 3]
others = [random.randint(2, 6) for _ in range(35)]  # total players = 6*8 + 2 extra

teams, final_gap = simulated_annealing_variable(captains, goalies, others)

teams = sort_teams(teams)

print("Final max team average gap:", round(final_gap, 4))
for i, team in enumerate(teams):
    print(f"Team {i+1} avg: {team_average(team):.3f}  -->  {team}")

Final max team average gap: 0.125
Team 1 avg: 4.000  -->  [5, 4, 2, 3, 4, 4, 5, 5]
Team 2 avg: 4.000  -->  [4, 3, 2, 3, 4, 5, 5, 6]
Team 3 avg: 4.125  -->  [5, 6, 2, 2, 3, 4, 5, 6]
Team 4 avg: 4.000  -->  [4, 4, 2, 3, 3, 5, 5, 6]
Team 5 avg: 4.000  -->  [3, 2, 3, 4, 4, 4, 6, 6]
Team 6 avg: 4.000  -->  [3, 3, 2, 4, 5, 5, 6]


In [23]:
teams, energy, best_goalie_assignment = best_goalie_permutation_sa(
    captains, goalies, others,
    T_start=1.0, T_end=1e-3, alpha=0.995, max_iters=100000
)

print("Best final gap after permuting goalies:", round(energy, 4))
print("Best goalie permutation:", best_goalie_assignment)

teams = sort_teams(teams)

for i, team in enumerate(teams):
    print(f"Team {i+1} avg: {team_average(team):.3f}  -->  {team}")

Energy: 0.125
Team 1 avg: 4.125  -->  [6, 3, 3, 4, 4, 4, 4, 5]
Team 2 avg: 4.125  -->  [3, 4, 3, 3, 4, 5, 5, 6]
Team 3 avg: 4.250  -->  [3, 4, 3, 3, 4, 5, 6, 6]
Team 4 avg: 4.125  -->  [3, 5, 3, 4, 4, 4, 4, 6]
Team 5 avg: 4.125  -->  [6, 2, 3, 3, 3, 4, 6, 6]
Team 6 avg: 4.143  -->  [4, 4, 3, 3, 5, 5, 5]
Energy: 0.125
Team 1 avg: 4.125  -->  [6, 3, 3, 3, 3, 4, 5, 6]
Team 2 avg: 4.250  -->  [3, 4, 3, 4, 4, 5, 5, 6]
Team 3 avg: 4.125  -->  [3, 4, 3, 4, 4, 4, 5, 6]
Team 4 avg: 4.125  -->  [3, 2, 3, 4, 5, 5, 5, 6]
Team 5 avg: 4.125  -->  [6, 4, 3, 3, 3, 4, 4, 6]
Team 6 avg: 4.143  -->  [4, 5, 3, 3, 4, 4, 6]
Energy: 0.125
Team 1 avg: 4.125  -->  [6, 4, 3, 3, 4, 4, 4, 5]
Team 2 avg: 4.250  -->  [3, 2, 3, 4, 4, 6, 6, 6]
Team 3 avg: 4.125  -->  [3, 4, 3, 3, 3, 5, 6, 6]
Team 4 avg: 4.125  -->  [3, 5, 3, 4, 4, 4, 5, 5]
Team 5 avg: 4.125  -->  [6, 3, 3, 3, 3, 4, 5, 6]
Team 6 avg: 4.143  -->  [4, 4, 3, 4, 4, 5, 5]
Energy: 0.125
Team 1 avg: 4.125  -->  [6, 4, 3, 3, 3, 3, 5, 6]
Team 2 avg: 4.125  -->

In [22]:
K = 6
captains = [6, 3, 3, 3, 6, 4]
goalies = [3, 4, 4, 5, 2, 4]
others = [3,3,4,5,5,6,3,4,4,4,6,3,3,4,4,5,6,3,4,4,5,6,6,3,3,4,5,5,4,3,3,4,5,6,3]
#Randomly permute others
#others = random.sample(others, len(others))
others = sorted(others)

teams, final_gap = simulated_annealing_variable(captains, goalies, others)

teams = sort_teams(teams)

print("Final max team average gap:", round(final_gap, 7))
for i, team in enumerate(teams):
    print(f"Team {i+1} avg: {team_average(team):.3f}  -->  {team}")

Final max team average gap: 0.125
Team 1 avg: 4.125  -->  [6, 3, 3, 3, 4, 4, 5, 5]
Team 2 avg: 4.125  -->  [3, 4, 3, 3, 4, 5, 5, 6]
Team 3 avg: 4.250  -->  [3, 4, 3, 3, 4, 5, 6, 6]
Team 4 avg: 4.125  -->  [3, 5, 3, 3, 4, 4, 5, 6]
Team 5 avg: 4.125  -->  [6, 2, 3, 3, 4, 4, 5, 6]
Team 6 avg: 4.143  -->  [4, 4, 3, 4, 4, 4, 6]


Energy: 0.125  
Team 1 avg: 4.125  -->  [6, 3, 3, 4, 4, 4, 4, 5]  
Team 2 avg: 4.125  -->  [3, 4, 3, 3, 4, 5, 5, 6]  
Team 3 avg: 4.250  -->  [3, 4, 3, 3, 4, 5, 6, 6]  
Team 4 avg: 4.125  -->  [3, 5, 3, 4, 4, 4, 4, 6]  
Team 5 avg: 4.125  -->  [6, 2, 3, 3, 3, 4, 6, 6]  
Team 6 avg: 4.143  -->  [4, 4, 3, 3, 5, 5, 5]  
